# Task 2 — CLM-8B trên Kaggle (Hướng 2, chạy đầy đủ)

1. Settings: Accelerator **GPU T4 x2**, Internet **On**.
2. Add-ons → Secrets: `PUBLIC_TEST_DRIVE_ID` (file id Google Drive của bộ dữ liệu, xin từ nhóm), tuỳ chọn `GITHUB_TOKEN` nếu repo private, `HF_TOKEN` nếu cần.
3. Tuỳ chọn: thêm Kaggle dataset chứa `silver_labels.json` (output của `scripts/run_task2_api.py` ở chế độ predict) và sửa `SILVER_PATH` bên dưới.
4. Run All. Dữ liệu sinh viên và repo nằm trong `/kaggle/tmp` (không được lưu cùng notebook version). Chỉ hai file được copy sang `/kaggle/working`: `task2_predictions.json` và `task2_benchmark.md`.

In [ ]:
REPO = 'github.com/tiendinhquang2104/llm-final-project.git'
BRANCH = 'task-2'
SILVER_PATH = ''  # e.g. '/kaggle/input/task2-silver/silver_labels.json'; '' = gold only
WORK = '/kaggle/tmp'  # student data + repo live here, not in /kaggle/working (which is saved with the notebook)

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
def secret(name):
    try:
        return secrets.get_secret(name)
    except Exception:
        return ''
token = secret('GITHUB_TOKEN')
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'
os.makedirs(WORK, exist_ok=True)
REPO_DIR = f'{WORK}/repo'
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone failed — check REPO/BRANCH/GITHUB_TOKEN') from None  # never print the token URL
    subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'], check=True)  # drop token from .git/config
os.chdir(REPO_DIR)
if secret('HF_TOKEN'):
    os.environ['HF_TOKEN'] = secret('HF_TOKEN')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q gdown 'transformers>=4.51,<5' accelerate safetensors scikit-learn pyyaml bitsandbytes

In [ ]:
import gdown
drive_id = secret('PUBLIC_TEST_DRIVE_ID')
assert drive_id, 'Add the PUBLIC_TEST_DRIVE_ID secret'
os.makedirs(f'{WORK}/download', exist_ok=True)
path = gdown.download(id=drive_id, output=f'{WORK}/download/data.zip', quiet=False)
assert path is not None, 'gdown failed — check PUBLIC_TEST_DRIVE_ID and sharing permissions'
!python scripts/unpack_data.py {WORK}/download --dest data/raw
!ls data/raw

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
args = ['--config', 'configs/task2/clm_kaggle.yaml'] + (['--silver', SILVER_PATH] if SILVER_PATH else [])
subprocess.run(['python', 'scripts/run_task2_clm.py', *args], check=True)

In [ ]:
!python scripts/task2_benchmark.py
!cp outputs/task2/task2_clm_kaggle/predictions.json /kaggle/working/task2_predictions.json
!cp outputs/task2/benchmark.md /kaggle/working/task2_benchmark.md
import json
from src.task2.evaluator import validate_predictions
sub = json.load(open('/kaggle/working/task2_predictions.json'))
print(len(sub), validate_predictions(sub))